In [ ]:
"""
Extrae las MFCC en secuencia (13 coef x 500 frames) para las 956 ventanas
del subconjunto balanceado del TFG. Salida: un .npz con X, etiquetas y folds.

El CSV ya trae, por fila, el archivo y el indice de ventana dentro de ese
archivo, asi que no hay que repetir el filtro de silencio ni el tope de 30
ventanas: se recorta directamente el segmento que toca.
"""

import os
import sys
import json
import numpy as np
import pandas as pd
import librosa

# --- Rutas ---
CSV_SUBCONJUNTO = r"C:\Users\josem\Desktop\tfg\dataset_60_topado.csv"
WAV_ROOT_DIRS = [r"C:\Users\josem\Desktop\tfg"]
OUT_NPZ = r"C:\Users\josem\Desktop\tfg\mfcc_secuencia_60.npz"

# --- Audio (heredado del TFG) ---
SR = 4000
WIN_SECONDS = 5
SAMPLES_PER_WINDOW = SR * WIN_SECONDS   # 20000

# --- MFCC ---
N_MFCC = 13
N_FFT = 256
WIN_LENGTH = 100        # 25 ms
HOP_LENGTH = 40         # 10 ms -> ~500 frames en 5 s
N_MELS = 40
FMIN = 0
FMAX = 2000
MAX_FRAMES = 500

CLASES = ["NORMAL", "ASTHMA", "BRON", "COPD", "HEART FAILURE", "PNEUMONIA"]


def construir_indice_wav(root_dirs):
    # Busca los .wav recursivamente y devuelve {nombre: ruta}.
    indice = {}
    for root in root_dirs:
        if not os.path.isdir(root):
            print(f"  aviso: no existe la carpeta {root}")
            continue
        for dirpath, _, files in os.walk(root):
            for f in files:
                if f.lower().endswith(".wav"):
                    indice[f] = os.path.join(dirpath, f)
    return indice


def mfcc_secuencia(y):
    m = librosa.feature.mfcc(
        y=y, sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT,
        hop_length=HOP_LENGTH, win_length=WIN_LENGTH,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX,
    ).T  # (frames, 13)

    n = m.shape[0]
    if n >= MAX_FRAMES:
        m = m[:MAX_FRAMES]
    else:
        pad = np.zeros((MAX_FRAMES - n, N_MFCC), dtype=m.dtype)
        m = np.vstack([m, pad])
    return m.astype(np.float32)


def main():
    if not os.path.isfile(CSV_SUBCONJUNTO):
        sys.exit(f"No encuentro el CSV: {CSV_SUBCONJUNTO}")

    df = pd.read_csv(CSV_SUBCONJUNTO, sep=";", decimal=",")
    df["ventana"] = df["ventana"].round().astype(int)
    print(f"{len(df)} ventanas, {df['pid'].nunique()} pacientes, "
          f"{df['archivo'].nunique()} archivos")

    indice_wav = construir_indice_wav(WAV_ROOT_DIRS)
    print(f"{len(indice_wav)} wav encontrados")

    faltan = sorted(a for a in df["archivo"].unique() if a not in indice_wav)
    if faltan:
        print(f"Faltan {len(faltan)} archivos, p.ej.: {faltan[:5]}")
        sys.exit("Revisa WAV_ROOT_DIRS")

    # Cada wav se carga una vez aunque tenga varias ventanas.
    cache = {}

    def cargar(archivo):
        if archivo not in cache:
            cache[archivo], _ = librosa.load(indice_wav[archivo], sr=SR)
        return cache[archivo]

    X = np.zeros((len(df), MAX_FRAMES, N_MFCC), dtype=np.float32)
    cortas = 0

    for i, fila in enumerate(df.itertuples(index=False)):
        senal = cargar(fila.archivo)
        ini = int(fila.ventana) * SAMPLES_PER_WINDOW
        seg = senal[ini:ini + SAMPLES_PER_WINDOW]

        if len(seg) < SAMPLES_PER_WINDOW:
            cortas += 1
            seg = np.pad(seg, (0, SAMPLES_PER_WINDOW - len(seg)))

        X[i] = mfcc_secuencia(seg)

        if (i + 1) % 100 == 0 or (i + 1) == len(df):
            print(f"  {i + 1}/{len(df)}", end="\r")
    print()

    if cortas:
        print(f"{cortas} ventanas venian cortas y se rellenaron con ceros")

    y_str = df["diagnostico"].to_numpy()
    desconocidas = sorted(set(y_str) - set(CLASES))
    if desconocidas:
        sys.exit(f"Clases inesperadas: {desconocidas}")
    clase_a_int = {c: k for k, c in enumerate(CLASES)}
    y_int = np.array([clase_a_int[c] for c in y_str], dtype=np.int64)

    pid = df["pid"].to_numpy()
    fold = df["fold"].to_numpy().astype(np.int64)
    archivo = df["archivo"].to_numpy()
    ventana = df["ventana"].to_numpy().astype(np.int64)

    np.savez_compressed(
        OUT_NPZ,
        X=X, y_str=y_str, y_int=y_int,
        pid=pid, fold=fold, archivo=archivo, ventana=ventana,
        clases=np.array(CLASES),
    )

    print(f"\nX: {X.shape}")
    for c in CLASES:
        print(f"  {c:14s}: {(y_str == c).sum()}")
    print(f"Guardado en {OUT_NPZ}")

    manifiesto = {
        "sr": SR, "win_seconds": WIN_SECONDS, "n_mfcc": N_MFCC, "n_fft": N_FFT,
        "win_length": WIN_LENGTH, "hop_length": HOP_LENGTH, "n_mels": N_MELS,
        "fmin": FMIN, "fmax": FMAX, "max_frames": MAX_FRAMES,
        "n_ventanas": int(len(df)), "clases": CLASES,
    }
    with open(os.path.splitext(OUT_NPZ)[0] + "_manifiesto.json", "w",
              encoding="utf-8") as fh:
        json.dump(manifiesto, fh, indent=2, ensure_ascii=False)


if __name__ == "__main__":
    main()